In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from aymurai.transforms.entity_subcategories.sentence_transformer import (
    SentenceTransformerSubcategorizer,
)
from aymurai.utils.json_data import load_json

sns.set_theme(style="whitegrid")
plt.rcParams.update(
    {"figure.figsize": (10, 6), "axes.titlesize": 14, "axes.labelsize": 12}
)

In [ ]:
# Config
pipeline_path = Path("/resources/pipelines/production/datapublic/pipeline.json")
annotations_path = Path(
    "/resources/annotations/label-studio/resos-annotations/30-nov/project-3-at-2022-11-30-16-04-2b43bf39.json"
)
target_labels = [
    "CONDUCTA",
    "CONDUCTA_DESCRIPCION",
    "DETALLE",
    "OBJETO_DE_LA_RESOLUCION",
]
encoder_name = "distiluse"
bm25_weight = 0.5  # use tuned default; set 0.0 for encoder-only
batch_size = 256
embeddings_dir = Path("/resources/cache/aymurai")
embeddings_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def collect_label_annotations(annotations, target_labels):
    buckets_by_label = defaultdict(list)
    target_labels = set(target_labels)

    for task in annotations:
        for annotation in task.get("annotations", []):
            merged = {}
            for result in annotation.get("result", []):
                result_id = result.get("id")
                value = result.get("value", {})
                slot = merged.setdefault(
                    result_id,
                    {
                        "text": value.get("text"),
                        "start": value.get("start"),
                        "end": value.get("end"),
                        "labels": [],
                        "choices": [],
                    },
                )
                slot["text"] = slot["text"] or value.get("text")
                slot["start"] = (
                    slot["start"] if slot["start"] is not None else value.get("start")
                )
                slot["end"] = (
                    slot["end"] if slot["end"] is not None else value.get("end")
                )
                slot["labels"].extend(value.get("labels", []))
                slot["choices"].extend(value.get("choices", []))

            for payload in merged.values():
                labels = [
                    label for label in payload["labels"] if label in target_labels
                ]
                if not labels or payload["text"] is None:
                    continue
                item = {
                    "text": payload["text"],
                    "labels": list(dict.fromkeys(payload["labels"])),
                    "choices": list(dict.fromkeys(payload["choices"])),
                    "start": payload["start"],
                    "end": payload["end"],
                }
                for label in labels:
                    buckets_by_label[label].append(item)
    return {label: buckets_by_label.get(label, []) for label in target_labels}


def compute_topk_accuracy(records, ks=(1, 2, 3, 4, 5)):
    metrics = []
    labels = sorted({record["label"] for record in records})
    for label in labels:
        label_records = [r for r in records if r["label"] == label]
        total = len(label_records)
        for k in ks:
            hits = sum(
                any(choice in r["retrieved"][:k] for choice in r["choices"])
                for r in label_records
            )
            metrics.append(
                {"label": label, "k": k, "accuracy": hits / total if total else np.nan}
            )
    overall = []
    for k in ks:
        hits = sum(
            any(choice in r["retrieved"][:k] for choice in r["choices"])
            for r in records
        )
        overall.append(
            {
                "label": "OVERALL",
                "k": k,
                "accuracy": hits / len(records) if records else np.nan,
            }
        )
    metrics.extend(overall)
    return pd.DataFrame(metrics)

In [ ]:
# Load annotations and collect samples
annotations = load_json(str(annotations_path))
samples_by_label = collect_label_annotations(annotations, target_labels)
# filter out empty-choice samples
for label in list(samples_by_label):
    samples_by_label[label] = [s for s in samples_by_label[label] if s.get("choices")]


def samples_to_dataframe(samples):
    rows = []
    for label, items in samples.items():
        for item in items:
            rows.append(
                {
                    "label": label,
                    "text": item["text"],
                    "choices": item["choices"],
                    "n_choices": len(item["choices"]),
                    "char_len": len(item["text"]),
                }
            )
    return pd.DataFrame(rows)


samples_df = samples_to_dataframe(samples_by_label)
samples_df.head()

In [ ]:
# Extract sentence-transformer config from pipeline (or fall back to targets)
with pipeline_path.open() as f:
    pipeline_config = json.load(f)

st_configs = {
    config[1]["category"]: {
        "embeddings_path": config[1].get("embeddings_path"),
    }
    for config in pipeline_config.get("postprocess", [])
    if "SentenceTransformerSubcategorizer" in config[0]
}

# Fallback: build configs from target_labels when pipeline does not define them
if not st_configs:
    st_configs = {
        label: {"embeddings_path": embeddings_dir / f"{label.lower()}.npz"}
        for label in target_labels
    }

st_configs

In [ ]:
# Build sentence-transformer subcategorizer models (hybrid by default)
models = {}
for label, cfg in st_configs.items():
    emb_path = cfg.get("embeddings_path") or (embeddings_dir / f"{label.lower()}.npz")
    models[label] = SentenceTransformerSubcategorizer(
        category=label,
        embeddings_path=str(emb_path),
        encoder_name=encoder_name,
        bm25_weight=bm25_weight,
        batch_size=batch_size,
        rebuild_embeddings=True,
    )
list(models.keys())

In [ ]:
# Re run to check loading existing embeddings
models = {}
for label, cfg in st_configs.items():
    emb_path = cfg.get("embeddings_path") or (embeddings_dir / f"{label.lower()}.npz")
    models[label] = SentenceTransformerSubcategorizer(
        category=label,
        embeddings_path=str(emb_path),
        encoder_name=encoder_name,
        bm25_weight=bm25_weight,
        batch_size=batch_size,
        rebuild_embeddings=False,
    )
list(models.keys())

In [ ]:
# Evaluate
records = []
for label, items in samples_by_label.items():
    retriever = models[label]
    texts = [item["text"] for item in items]
    retrieved_lists = retriever.batch_retrieve(texts, top_k=5)
    for item, retrieved in zip(items, retrieved_lists):
        records.append(
            {
                "label": label,
                "text": item["text"],
                "choices": item["choices"],
                "retrieved": retrieved,
            }
        )

accuracy_df = compute_topk_accuracy(records)
pivot = accuracy_df.pivot(index="label", columns="k", values="accuracy").sort_index()
pivot

In [ ]:
# Heatmap
fig, ax = plt.subplots()
sns.heatmap(
    pivot.loc[[label for label in pivot.index if label != "OVERALL"]],
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    ax=ax,
)
ax.set_title("SentenceTransformerSubcategorizer (distiluse) top-k accuracy")
ax.set_xlabel("k")
ax.set_ylabel("Label")
plt.tight_layout()
plt.show()

pivot.loc[["OVERALL"]]

In [ ]:
encoder_name = "minilm"

# Build sentence-transformer subcategorizer models (hybrid by default)
models = {}
for label, cfg in st_configs.items():
    emb_path = cfg.get("embeddings_path") or (embeddings_dir / f"{label.lower()}.npz")
    models[label] = SentenceTransformerSubcategorizer(
        category=label,
        embeddings_path=str(emb_path),
        encoder_name=encoder_name,
        bm25_weight=bm25_weight,
        batch_size=batch_size,
        rebuild_embeddings=True,
    )
list(models.keys())

In [ ]:
# Evaluate
records = []
for label, items in samples_by_label.items():
    retriever = models[label]
    texts = [item["text"] for item in items]
    retrieved_lists = retriever.batch_retrieve(texts, top_k=5)
    for item, retrieved in zip(items, retrieved_lists):
        records.append(
            {
                "label": label,
                "text": item["text"],
                "choices": item["choices"],
                "retrieved": retrieved,
            }
        )

accuracy_df = compute_topk_accuracy(records)
pivot = accuracy_df.pivot(index="label", columns="k", values="accuracy").sort_index()
pivot

In [ ]:
# Heatmap
fig, ax = plt.subplots()
sns.heatmap(
    pivot.loc[[label for label in pivot.index if label != "OVERALL"]],
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    ax=ax,
)
ax.set_title("SentenceTransformerSubcategorizer (minilm) top-k accuracy")
ax.set_xlabel("k")
ax.set_ylabel("Label")
plt.tight_layout()
plt.show()

pivot.loc[["OVERALL"]]